# Customer Churn Prediction & Retention Analytics
**Portfolio project | Python · Pandas · Scikit-learn · Matplotlib · Seaborn**

## Business problem
Subscription businesses lose revenue when customers cancel. This notebook explores churn patterns, trains baseline machine-learning models, evaluates their results, and creates a ranked review list for supportive retention outreach.

## Dataset
Use an authorized copy of the IBM Telco Customer Churn sample dataset from Kaggle or another source whose terms permit your use. Attach the CSV using Kaggle's **Add Input** option, then update `DATA_PATH` below to the exact path shown in the Input panel. This project does not include customer data.

## Responsible use
Risk scores are estimates, not certainty. This is a learning prototype, not a production decision system. Do not use scores to deny service or set unfair prices. Review subgroup performance, privacy, and human oversight before real deployment.

## 1. Import libraries and configure the dataset path

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
DATA_PATH = Path("/kaggle/input/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv")
TARGET = "Churn"

print("Ready. Update DATA_PATH if the CSV is stored in another Kaggle folder.")

## 2. Load and inspect data

In [ ]:
if not DATA_PATH.exists():
    print("CSV not found at:", DATA_PATH)
    print("Attach the dataset with Add Input and update DATA_PATH to the path shown by Kaggle.")
else:
    df = pd.read_csv(DATA_PATH)
    print("Shape:", df.shape)
    display(df.head())
    display(df.dtypes.to_frame("dtype"))
    display(df.isna().sum().sort_values(ascending=False).head(15).to_frame("missing_count"))

## 3. Clean and validate the data

In [ ]:
if DATA_PATH.exists():
    df.columns = [c.strip() for c in df.columns]
    if TARGET not in df.columns:
        raise ValueError(f"Target '{TARGET}' not found. Available columns: {df.columns.tolist()}")

    # Customer ID identifies records but should not be a predictive feature.
    if "customerID" in df.columns:
        df = df.drop(columns=["customerID"])
    if "TotalCharges" in df.columns:
        df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

    df[TARGET] = df[TARGET].astype(str).str.strip()
    df = df[df[TARGET].isin(["Yes", "No"])].copy()
    df["ChurnFlag"] = (df[TARGET] == "Yes").astype(int)

    print("Cleaned shape:", df.shape)
    print(f"Observed churn rate: {df['ChurnFlag'].mean():.1%}")
    display(df[TARGET].value_counts().rename_axis("Churn").to_frame("customers"))

## 4. Exploratory data analysis

In [ ]:
if DATA_PATH.exists():
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    sns.countplot(data=df, x=TARGET, ax=axes[0])
    axes[0].set_title("Customer churn distribution")
    sns.histplot(data=df, x="tenure", hue=TARGET, bins=25, multiple="stack", ax=axes[1])
    axes[1].set_title("Tenure distribution by churn")
    plt.tight_layout()
    plt.show()

    for col in ["Contract", "InternetService", "PaymentMethod"]:
        if col in df.columns:
            rates = (df.groupby(col)["ChurnFlag"].mean() * 100).sort_values(ascending=False)
            display(rates.round(1).rename("churn_rate_percent").to_frame())

## 5. Split data and build preprocessing pipeline
Imputation, scaling, and one-hot encoding are fitted inside a pipeline to reduce leakage. The test set is held out for evaluation.

In [ ]:
if DATA_PATH.exists():
    X = df.drop(columns=[TARGET, "ChurnFlag"], errors="ignore")
    y = df["ChurnFlag"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )

    numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
    categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])

    print("Train rows:", len(X_train), "| Test rows:", len(X_test))
    print("Numeric features:", len(numeric_features))
    print("Categorical features:", len(categorical_features))

## 6. Train and compare models

In [ ]:
if DATA_PATH.exists():
    models = {
        "Dummy baseline": DummyClassifier(strategy="most_frequent"),
        "Logistic regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
        "Random forest": RandomForestClassifier(
            n_estimators=250, min_samples_leaf=3, class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=-1
        )
    }
    fitted_models = {}
    results = []

    for name, estimator in models.items():
        pipe = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
        pipe.fit(X_train, y_train)
        fitted_models[name] = pipe
        pred = pipe.predict(X_test)
        prob = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, "predict_proba") else None
        results.append({
            "model": name,
            "accuracy": accuracy_score(y_test, pred),
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred, zero_division=0),
            "f1": f1_score(y_test, pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, prob) if prob is not None else np.nan
        })

    metrics_df = pd.DataFrame(results).sort_values("f1", ascending=False)
    display(metrics_df.style.format({c: "{:.3f}" for c in ["accuracy", "precision", "recall", "f1", "roc_auc"]}))
    print("Compare recall and precision as well as accuracy; the classes may be imbalanced.")

## 7. Evaluate the selected model
For a simple learning demonstration, this selects the highest-F1 model in the comparison. For a rigorous project, choose a model using validation data or cross-validation, then evaluate the final test set only once.

In [ ]:
if DATA_PATH.exists():
    selected_name = metrics_df.iloc[0]["model"]
    selected_model = fitted_models[selected_name]
    y_pred = selected_model.predict(X_test)
    y_prob = selected_model.predict_proba(X_test)[:, 1] if hasattr(selected_model, "predict_proba") else None

    print("Selected model:", selected_name)
    print(classification_report(y_test, y_pred, target_names=["No churn", "Churn"], zero_division=0))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Predicted no", "Predicted churn"],
                yticklabels=["Actual no", "Actual churn"])
    plt.title("Confusion matrix")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.show()

    if y_prob is not None:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[0])
        PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[1])
        axes[0].set_title("ROC curve")
        axes[1].set_title("Precision-recall curve")
        plt.tight_layout()
        plt.show()

## 8. Create a churn-risk review list

In [ ]:
if DATA_PATH.exists() and y_prob is not None:
    risk_review = X_test.copy()
    risk_review["actual_churn"] = y_test.values
    risk_review["predicted_churn_probability"] = y_prob
    risk_review["predicted_churn"] = y_pred
    risk_review = risk_review.sort_values("predicted_churn_probability", ascending=False)

    display(risk_review.head(20))
    risk_review.to_csv("customer_churn_risk_review.csv", index=False)
    print("Saved customer_churn_risk_review.csv")

## 9. Export metrics and reflect on business impact

In [ ]:
if DATA_PATH.exists():
    metrics_df.to_csv("model_comparison_metrics.csv", index=False)
    print("Saved model_comparison_metrics.csv")
    print("Questions to answer in your project report:")
    print("1. Which model has the best balance of precision and recall?")
    print("2. How many churners were detected and missed?")
    print("3. What helpful retention outreach might be appropriate?")
    print("4. How would you check performance across customer groups?")
    print("5. How would you monitor model drift over time?")

## 10. Conclusions and limitations
After running the notebook, write conclusions based on your actual metrics. Do not invent or claim results before executing the analysis.

Limitations:
- Sample data may not represent a current real business.
- Association does not prove causation.
- Performance can change over time and needs monitoring.
- This notebook is not a production API and has no authentication or access controls.
- Risk scores should support helpful outreach, not punitive decisions.

**Suggested GitHub description:** End-to-end customer churn analytics and machine learning project using Python, scikit-learn, and retention risk analysis.